# Cuaderno 01 · Inspección de estructura y calidad de datos

**Expediente:** BES-jun · **Cuaderno:** 2 de 7 · **Estado del expediente:** Listo para revisión (no Completo)

## 1. Introducción y contexto (el problema)
El libro no es una tabla: es un **formulario de presentación** (celdas combinadas, rótulos y valores mezclados, sin fila de encabezado) con 1 270 fórmulas en cinco hojas. Una lectura por posición fija funciona en una hoja y falla **sin avisar** en otra, porque las filas se desplazan entre hojas. Un valor leído de la celda equivocada es un número plausible pero falso que entra a la cadena de cálculo.

## 2. Justificación y necesidad
Antes de transcribir cualquier fórmula hay que saber: dónde está cada dato en cada hoja, qué defectos trae el libro (errores guardados, fechas que cambian solas, datos faltantes) y si los datos geométricos (survey, profundidades de equipos) son internamente coherentes. Hay que poder decir, con evidencia, qué defectos **sí** afectan a los cálculos y cuáles no.

## 3. Objetivo general
Emitir un dictamen de calidad por hoja, apoyado en controles ejecutables y reproducibles, que distinga defectos que contaminan la cadena de cálculo de defectos que no.

## 4. Dominio y alcance
**Incluye:** inventario estructural; anclaje por rótulo y demostración del riesgo de la coordenada fija; datos faltantes; errores cacheados y rastreo de dependencias; fechas; geometría del survey y contraste con el método de curvatura mínima; coherencia de las TVD de equipos con el survey; controles de dominio sobre las entradas.

**No incluye:** corregir o imputar valores (no se modifica nada); interpretar la física de las fórmulas (cuaderno 02); juzgar si las condiciones de operación son «buenas» o «malas» (no hay límites aprobados).

## 5. Fuentes y datos (entradas y salidas)
**Entradas:** `entrada/BES_jun.xlsx` (copia verificada en el cuaderno 00). **Salidas:** tablas de inventario, anclas, errores, precedentes, survey, TVD y plausibilidad; dictamen por hoja.

## 6. Objetivos específicos y criterios de aceptación
**C01-1** 5 hojas visibles y 1 270 fórmulas inventariadas · **C01-2** todos los rótulos clave hallados en todas las hojas · **C01-3** ningún error guardado es precedente de THD ni de «Caudal a producir» · **C01-4** cero datos requeridos faltantes · **C01-5** geometría del survey correcta en las 5 hojas · **C01-6** survey = curvatura mínima dentro de 0.01 ft / 0.01 °/100 ft · **C01-7** TVD de equipos = interpolación del survey (< 1e-6 ft) · **C01-8** ningún control de dominio incumplido · **C01-9** el libro no cambió durante el cuaderno.

## 7. Riesgo si este paso falla
Una entrada mal leída o un error guardado que entra a la cadena produce PIP, THD y caudales con apariencia normal. Si además el survey fuera inconsistente, todas las profundidades de equipos —y con ellas la presión en el intake— quedarían sesgadas sin señal visible.

## Preparación del entorno
La celda siguiente es la misma en todos los cuadernos: localiza el expediente, habilita `src/` y registra la ficha de reproducibilidad (fecha, versiones y huella del insumo).

In [1]:
from pathlib import Path
import os, sys

def localizar_expediente() -> Path:
    # Sube desde el directorio actual hasta la carpeta que tiene dataset.md y entrada/.
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "dataset.md").is_file() and (p / "entrada").is_dir():
            return p
    raise RuntimeError("Ejecute el cuaderno dentro del expediente (donde están dataset.md y entrada/).")

RAIZ = localizar_expediente()                 # .../datasets/en_uso/BES-jun
RAIZ_PROYECTO = RAIZ.parents[2]               # .../HUC
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda v: f"{v:,.8g}")

from analisis.criterios import Criterios
from analisis.entorno import ficha
from analisis.presentacion import T
T(pd.DataFrame(list(ficha().items()), columns=["campo", "valor"]))

campo,valor
fecha_ejecucion,2026-10-20T14:25:36-03:00
python,3.14.8
sistema,Linux 7.2.10-100.fc43.x86_64
pandas,3.0.6
numpy,2.5.4
matplotlib,3.11.2
openpyxl,3.1.5
expediente,BES-jun
insumo,entrada/BES_jun.xlsx
insumo_sha256,ccb5a738baea13321993e8b6eccefd6cac90ecda86b1c4b9181f897fcfa83f85


In [2]:
from ingesta.integridad import huellas
from ingesta.leer_bes_jun import COPIA_TRABAJO, leer_libro
sha_inicial = huellas(COPIA_TRABAJO)["sha256"]      # se compara de nuevo al final (C01-9)
hojas = leer_libro()                                # lectura sin guardar; abre el libro 2 veces (fórmulas y caché)
print("Hojas leídas:", [h["hoja"] for h in hojas])

Hojas leídas: ['QF-554H', 'QF-545H', 'QF-500H', 'QF-378HST', 'QF-549HST']


## Paso 1 · Inventario estructural

**Qué se hace.** Se recorre cada celda y se clasifica como fórmula, constante o error guardado; se cuentan celdas combinadas y se listan las funciones de Excel usadas. **Por qué.** Sirve para dimensionar el problema: un libro con 1 270 fórmulas y casi 5 000 constantes no puede revisarse a ojo. También responde una pregunta de aislamiento: si alguna fórmula citara **otra hoja**, un cambio en una hoja contaminaría a otra y no podríamos tratarlas por separado.

In [3]:
from ingesta.estructura_libro import inventario_hojas
inv = pd.DataFrame(inventario_hojas())
inv["usa TODAY()"] = inv["funciones"].str.contains("TODAY")
print("Fórmulas totales:", int(inv["formulas"].sum()), "| constantes totales:", int(inv["constantes"].sum()),
      "| fórmulas que citan otra hoja:", int(inv["formulas_que_citan_otra_hoja"].sum()))
T(inv[["hoja", "estado", "dimension", "celdas_combinadas", "formulas", "constantes", "errores_en_cache", "usa TODAY()"]])

Fórmulas totales: 1270 | constantes totales: 4885 | fórmulas que citan otra hoja: 0


hoja,estado,dimension,celdas_combinadas,formulas,constantes,errores_en_cache,usa TODAY()
QF-554H,visible,A1:S388,82,261,973,0,sí
QF-545H,visible,A1:S382,80,240,979,0,sí
QF-500H,visible,A1:S357,80,260,1031,0,sí
QF-378HST,visible,A1:S343,79,263,967,0,sí
QF-549HST,visible,A1:S351,79,246,935,13,no


In [4]:
T(inv[["hoja", "funciones"]])

hoja,funciones
QF-554H,"AVERAGE×18, IF×9, INDEX×28, MATCH×7, SQRT×19, TODAY×1"
QF-545H,"IF×9, INDEX×28, MATCH×7, SQRT×19, TODAY×1"
QF-500H,"AVERAGE×18, IF×9, INDEX×28, MATCH×7, SQRT×19, TODAY×1"
QF-378HST,"AVERAGE×18, IF×9, INDEX×28, MATCH×7, SQRT×18, TODAY×1"
QF-549HST,"AVERAGE×9, IF×9, INDEX×28, MATCH×7, SQRT×16"


**Interpretación.**
- Las cinco hojas están **visibles** y ninguna fórmula cita otra hoja: cada hoja es un cálculo **autónomo** y puede tratarse por separado sin riesgo de contaminación cruzada. (No implica que sean independientes entre sí en sus datos: comparten tablas de bomba, ver cuaderno 06.)
- Las funciones son pocas y elementales (`INDEX`, `MATCH`, `IF`, `SQRT`, `AVERAGE`, `TODAY`). `MATCH`/`INDEX` implementan las interpolaciones de profundidad y la intersección con la curva de bomba; `TODAY` es la fecha del encabezado.
- Cuatro de las cinco hojas usan `TODAY()`; la restante (QF-549HST) tiene una fecha escrita. Se detalla en el Paso 5.
- `QF-549HST` es la única con errores guardados (13). El Paso 4 demuestra si importan.

## Paso 2 · Por qué las celdas se buscan por rótulo y no por coordenada

**Hipótesis a poner a prueba.** Las cinco hojas parecen la misma plantilla, así que sería tentador leer, por ejemplo, el caudal de operación siempre de `B71`. Se contrasta esa idea con los hechos: se busca cada rótulo clave en cada hoja (`anclas_por_hoja`) y después se compara lo que leería la coordenada fija `B71` contra el valor correcto.

In [5]:
from ingesta.estructura_libro import anclas_por_hoja
anclas = pd.DataFrame(anclas_por_hoja()).rename(columns={
    "Descarga (mecánica)": "Descarga", "Intake (mecánica)": "Intake", "Sensor (mecánica)": "Sensor",
    "Velocidad (operación)": "Velocidad", "Caudal (operación)": "Caudal", "Presión de Sensor": "P sensor",
    "PIP en (ft)": "PIP ft", "THD TOTAL": "THD", "Limite maximo": "Lím. máx", "Caudal a producir": "Q producir"})
print("Fila donde está cada rótulo en cada hoja:")
T(anclas)

Fila donde está cada rótulo en cada hoja:


hoja,Descarga,Intake,Sensor,Velocidad,Caudal,P sensor,PIP ft,THD,Lím. máx,Q producir
QF-554H,18,20,24,69,71,73,96,102,58,106
QF-545H,17,19,23,67,69,71,94,100,57,104
QF-500H,17,19,23,68,70,72,95,101,57,105
QF-378HST,17,19,23,66,68,70,93,99,57,103
QF-549HST,17,19,23,66,68,70,93,99,57,103


In [6]:
from openpyxl import load_workbook
wb = load_workbook(COPIA_TRABAJO, data_only=True)
demo = []
for h in hojas:
    ws = wb[h["hoja"]]
    leido = ws["B71"].value
    demo.append({"hoja": h["hoja"], "rótulo en A71": ws["A71"].value, "valor en B71": leido,
                 "caudal correcto (por rótulo)": h["caudal_operacion_bfpd"],
                 "¿coincide?": leido == h["caudal_operacion_bfpd"]})
wb.close()
print("Lo que habría leído un acceso por coordenada fija B71:")
T(pd.DataFrame(demo))

Lo que habría leído un acceso por coordenada fija B71:


hoja,rótulo en A71,valor en B71,caudal correcto (por rótulo),¿coincide?
QF-554H,Caudal,10301,10301,sí
QF-545H,Presión de Sensor,667,8767,no
QF-500H,—,—,7728,no
QF-378HST,—,—,7116,no
QF-549HST,—,—,5419,no


**Interpretación.** Las filas de los bloques **cambian entre hojas** (p. ej. «Caudal a producir» está en las filas 106, 104, 105, 103 y 103). La coordenada fija `B71` solo acierta en la primera hoja; en las otras devolvería el valor de **otra variable** o una celda vacía, y lo haría sin ningún error. Esta es la justificación técnica del diseño del lector: busca por **texto de rótulo**, y si un rótulo no aparece devuelve «faltante» en vez de adivinar. El cuaderno 03 deja registrada, para cada variable, la celda concreta de la que salió.

## Paso 3 · Datos requeridos: ¿falta algo?

Un cálculo con una entrada ausente debe **declararse no calculable**, no completarse con un supuesto. Se verifica que las 19 variables que consumen las fórmulas del expediente estén presentes en las cinco hojas, y que la tabla de la bomba tenga puntos.

In [7]:
requeridas = ["frecuencia_hz", "caudal_operacion_bfpd", "presion_sensor_psi", "thp_psi", "id_tubing_in", "etapas",
              "api_deg", "bsw_numero", "temperatura_c", "densidad_agua_lb_gal", "tvd_descarga_ft", "tvd_intake_ft",
              "tvd_sensor_ft", "frecuencia_curva_hz", "limite_minimo_bfpd", "limite_maximo_bfpd",
              "cache_pip_ft", "cache_thd_ft", "cache_caudal_producir_bfpd"]
completitud = []
for h in hojas:
    faltan = [k for k in requeridas if h.get(k) is None]
    completitud.append({"hoja": h["hoja"], "requeridas": len(requeridas),
                        "presentes": len(requeridas) - len(faltan),
                        "faltantes": ", ".join(faltan) or "ninguna",
                        "puntos en tabla de bomba": len(h["curva"])})
T(pd.DataFrame(completitud))

hoja,requeridas,presentes,faltantes,puntos en tabla de bomba
QF-554H,19,19,ninguna,6
QF-545H,19,19,ninguna,6
QF-500H,19,19,ninguna,6
QF-378HST,19,19,ninguna,6
QF-549HST,19,19,ninguna,6


**Interpretación.** No falta ninguna entrada requerida y las tablas de bomba tienen seis puntos en todas las hojas. Esto **no** significa que los valores sean correctos, solo que existen. La correctitud física se aborda en los Pasos 6 a 8.

## Paso 4 · Errores guardados y rastreo de dependencias

**Hecho.** Excel guarda el resultado de cada fórmula; si una fórmula divide por cero, guarda `#DIV/0!`. **Pregunta de ingeniería.** ¿Esos errores contaminan lo que calculamos (PIP, THD, caudal a producir) o están en otra parte de la hoja? Afirmar «no importan» sin demostrarlo sería un riesgo; por eso se **rastrean las precedencias**: a partir de la celda resultado se recorre, fórmula por fórmula, todo de lo que depende (`precedentes`), y se verifica si alguna celda con error aparece en ese conjunto.

In [8]:
from ingesta.estructura_libro import errores_cacheados
errores = pd.DataFrame(errores_cacheados())
print("Errores guardados en el libro:", len(errores), "| por hoja:", errores["hoja"].value_counts().to_dict())
T(errores)

Errores guardados en el libro: 13 | por hoja: {'QF-549HST': 13}


hoja,celda,valor_en_cache,formula
QF-549HST,D237,#DIV/0!,=AVERAGE(D234:D236)
QF-549HST,D241,#DIV/0!,=AVERAGE(D238:D240)
QF-549HST,D245,#DIV/0!,=AVERAGE(D242:D244)
QF-549HST,D246,#DIV/0!,=D245*D237*SQRT(3)/1000/0.93
QF-549HST,D251,#DIV/0!,=AVERAGE(D248:D250)
QF-549HST,D255,#DIV/0!,=AVERAGE(D252:D254)
QF-549HST,D259,#DIV/0!,=AVERAGE(D256:D258)
QF-549HST,D260,#DIV/0!,=D251*D259*SQRT(3)/1000
QF-549HST,D265,#DIV/0!,=AVERAGE(D262:D264)
QF-549HST,D269,#DIV/0!,=AVERAGE(D266:D268)


In [9]:
from analisis.dependencias import precedentes
wf = load_workbook(COPIA_TRABAJO, data_only=False)
con_error = errores.groupby("hoja")["celda"].apply(set).to_dict()
filas = []
for a in anclas_por_hoja():
    ws = wf[a["hoja"]]
    for nombre, celda in (("THD TOTAL", f"G{a['THD TOTAL']}"), ("Caudal a producir", f"B{a['Caudal a producir']}")):
        p = precedentes(ws, celda)
        comunes = (p["formulas"] | p["constantes"]) & con_error.get(a["hoja"], set())
        filas.append({"hoja": a["hoja"], "resultado": nombre, "celda": celda,
                      "fórmulas precedentes": len(p["formulas"]), "constantes precedentes": len(p["constantes"]),
                      "cita otra hoja": p["cita_otra_hoja"], "errores entre precedentes": len(comunes)})
wf.close()
rastreo = pd.DataFrame(filas)
T(rastreo)

hoja,resultado,celda,fórmulas precedentes,constantes precedentes,cita otra hoja,errores entre precedentes
QF-554H,THD TOTAL,G102,33,123,no,0
QF-554H,Caudal a producir,B106,54,138,no,0
QF-545H,THD TOTAL,G100,33,147,no,0
QF-545H,Caudal a producir,B104,54,162,no,0
QF-500H,THD TOTAL,G101,34,152,no,0
QF-500H,Caudal a producir,B105,55,167,no,0
QF-378HST,THD TOTAL,G99,34,128,no,0
QF-378HST,Caudal a producir,B103,55,143,no,0
QF-549HST,THD TOTAL,G99,33,145,no,0
QF-549HST,Caudal a producir,B103,54,160,no,0


**Interpretación.** Los 13 errores son `#DIV/0!` y están todos en `QF-549HST`, en la columna D de un bloque lejano a la cadena de cálculo. El rastreo muestra que **ninguna** de las celdas de las que dependen THD y «Caudal a producir» es una celda con error, en ninguna hoja. Por tanto estos errores son un defecto **del libro** (conviene avisarlo a quien lo mantiene) pero **no contaminan** los cálculos de este expediente. La conclusión se limita a eso: no se afirma nada sobre el bloque donde están los errores, que queda fuera del alcance.

*Límite del método:* el rastreo lee referencias `A1` escritas en la fórmula (con rangos); no evalúa funciones ni sigue nombres definidos (el libro no tiene) ni otras hojas (ninguna fórmula las cita, Paso 1).

## Paso 5 · Fechas del encabezado

Una fecha en el encabezado de un ensayo de pozo suele indicar **cuándo** se tomó el punto. Aquí hay que comprobar si realmente lo hace, porque de eso depende si se puede ordenar los datos en el tiempo.

In [10]:
T(pd.DataFrame([{"hoja": h["hoja"], "fórmula/valor en la celda de fecha": h["fecha_formula"],
                 "valor guardado": h["fecha_cache"], "es TODAY()": h["fecha_es_today"]} for h in hojas]))

hoja,fórmula/valor en la celda de fecha,valor guardado,es TODAY()
QF-554H,=TODAY(),2026-10-17 00:00:00,sí
QF-545H,=TODAY(),2026-10-17 00:00:00,sí
QF-500H,=TODAY(),2026-10-17 00:00:00,sí
QF-378HST,=TODAY(),2026-10-17 00:00:00,sí
QF-549HST,2017-09-06 00:00:00,2017-09-06 00:00:00,no


**Interpretación.** En cuatro hojas la fecha es `=TODAY()`: es la **fecha en que Excel recalculó**, no la fecha del ensayo. Cambia cada día que se abre el libro. En la quinta hoja hay una fecha escrita (2017-09-06) sin que el libro explique a qué corresponde. **Consecuencia:** el libro **no permite análisis temporal** ni ordenar los puntos por fecha. Cada hoja es un punto de operación aislado.

## Paso 6 · Coherencia geométrica del survey

El survey (tabla de estaciones MD, inclinación, azimut, TVD y *dogleg*) define las profundidades verticales que usa la hoja para corregir presiones. Si fuera incoherente, el error se propagaría a todo. Se hacen **dos tipos de control**:

1. **Propiedades geométricas** que cualquier survey debe cumplir: MD estrictamente creciente y sin duplicados; TVD ≤ MD; inclinación en [0°, 180°]; azimut en [0°, 360°]; sin nulos.
2. **Contraste independiente.** Se recalculan TVD y *dogleg* con el **método de curvatura mínima**, procedimiento estándar de la industria, y se comparan con las columnas de la hoja. *Importante:* ese método **no es una fórmula de este proyecto**; se usa solo como control externo. La tolerancia es 0.01, una unidad del último decimal que muestra la hoja; no es un límite de ingeniería.

In [11]:
from ingesta.survey import leer_survey
from analisis.calidad import verificar_survey, contrastar_con_curvatura_minima
estaciones = leer_survey()
geom, mcm = [], []
for h in hojas:
    est = [e for e in estaciones if e["hoja"] == h["hoja"]]
    geom.append({"hoja": h["hoja"], **verificar_survey(est)})
    mcm.append({"hoja": h["hoja"], **contrastar_con_curvatura_minima(est)})
geom = pd.DataFrame(geom)
mcm = pd.DataFrame(mcm)
T(geom[["hoja", "estaciones", "md_final_ft", "md_estrictamente_creciente", "md_duplicadas",
        "vertical_menor_o_igual_que_md", "vertical_monotona", "inclinacion_max_deg", "azimut_en_0_360", "valores_nulos"]]
  .rename(columns={"md_estrictamente_creciente": "MD creciente", "md_duplicadas": "MD duplic.",
                   "vertical_menor_o_igual_que_md": "TVD≤MD", "vertical_monotona": "TVD monótona",
                   "inclinacion_max_deg": "inc. máx (°)", "azimut_en_0_360": "azim. 0-360", "valores_nulos": "nulos",
                   "md_final_ft": "MD final (ft)"}), ancha=True)

hoja,estaciones,MD final (ft),MD creciente,MD duplic.,TVD≤MD,TVD monótona,inc. máx (°),azim. 0-360,nulos
QF-554H,56,4530,sí,0,sí,no,90.68,sí,0
QF-545H,68,4576,sí,0,sí,no,91.5,sí,0
QF-500H,71,4637,sí,0,sí,no,90.8,sí,0
QF-378HST,59,4598,sí,0,sí,no,90.49,sí,0
QF-549HST,67,4592,sí,0,sí,no,91.48,sí,0


In [12]:
T(mcm.rename(columns={"max_dif_tvd_ft": "máx |ΔTVD| (ft)", "max_dif_dogleg_deg_100ft": "máx |Δdogleg| (°/100 ft)",
                      "coincide_dentro_del_redondeo": "coincide"}))

hoja,máx |ΔTVD| (ft),máx |Δdogleg| (°/100 ft),tolerancia,coincide
QF-554H,0.0049496518,0.0049214971,0.01,sí
QF-545H,0.0047268996,0.0048730872,0.01,sí
QF-500H,0.0049282311,0.0049082785,0.01,sí
QF-378HST,0.0049141367,0.0049929494,0.01,sí
QF-549HST,0.0049738155,0.0048000848,0.01,sí


**Interpretación.**
- La geometría cumple en las cinco hojas: MD estrictamente creciente, sin duplicados ni nulos, TVD ≤ MD.
- **«TVD monótona = no»** no es un defecto: con inclinaciones por encima de 90° (hasta ≈ 91.5° en algunas hojas) el pozo puede subir ligeramente y la profundidad vertical decrece en tramos. Una regla «TVD siempre creciente» habría dado una **falsa alarma**.
- Las columnas **Vertical** y **Dogleg** de la hoja coinciden con el recálculo por curvatura mínima con diferencias por debajo de 0.005 (el redondeo a dos decimales de la hoja). Esto cierra un pendiente anterior (el método con que se obtuvo `Vertical` no estaba verificado): ahora se sabe que es **coherente con curvatura mínima**.
- *Límite:* es coherencia interna; no demuestra que el survey medido en campo sea exacto.

## Paso 7 · ¿De dónde salen las TVD de descarga, intake y sensor?

La hoja escribe MD de cada equipo y calcula la TVD con `MATCH`/`INDEX` sobre el survey. Se **reproduce** esa interpolación en Python (`np.interp`, lineal) y se compara con el valor que guarda la hoja. Además se exige que la MD del equipo esté **dentro del rango del survey** (extrapolar sería un error silencioso) y que TVD ≤ MD.

In [13]:
from analisis.calidad import verificar_tvd_de_equipos
tvd_equipos = pd.DataFrame([f for h in hojas for f in verificar_tvd_de_equipos(
    h, [e for e in estaciones if e["hoja"] == h["hoja"]])])
T(tvd_equipos.rename(columns={"md_ft": "MD (ft)", "tvd_hoja_ft": "TVD hoja (ft)", "tvd_interpolada_ft": "TVD interpolada (ft)",
                              "dif_ft": "dif (ft)", "dentro_del_survey": "MD dentro del survey",
                              "tvd_menor_o_igual_que_md": "TVD≤MD"}))

hoja,equipo,MD (ft),TVD hoja (ft),TVD interpolada (ft),dif (ft),MD dentro del survey,TVD≤MD
QF-554H,descarga,2841.51,2644.7775,2644.7775,0,sí,sí
QF-554H,intake,2856.1,2653.1767,2653.1767,0,sí,sí
QF-554H,sensor,2900.58,2678.822,2678.822,0,sí,sí
QF-545H,descarga,2978.66,2527.362,2527.362,0,sí,sí
QF-545H,intake,2994.87,2534.6038,2534.6038,0,sí,sí
QF-545H,sensor,3038.12,2553.9174,2553.9174,0,sí,sí
QF-500H,descarga,2801.9,2572.4604,2572.4604,0,sí,sí
QF-500H,intake,2813.57,2578.9682,2578.9682,0,sí,sí
QF-500H,sensor,2859.3,2604.4984,2604.4984,0,sí,sí
QF-378HST,descarga,3018.77,2763.6426,2763.6426,0,sí,sí


**Interpretación.** Las 15 TVD (3 equipos × 5 hojas) coinciden con la interpolación lineal del survey con diferencia nula, todas las MD están dentro del rango y TVD ≤ MD. Por tanto las profundidades que usan PIP, THP y fricción **provienen del survey** y se pueden reproducir de forma independiente. *Límite:* se verifica la interpolación, no que las MD de los equipos sean las reales de la completación.

## Paso 8 · Controles de dominio sobre las entradas

Los controles `D-xx` son condiciones **sin las cuales la fórmula transcrita deja de tener sentido** (p. ej. `°API > −131.5` porque la fórmula divide por `131.5 + °API`; `BS&W ∈ [0, 1]` porque la fórmula lo multiplica como fracción). Los `G-xx` son geometría básica de la sarta. El `O-01` es una **observación**, no un juicio: compara la presión del sensor con la de burbuja, porque la cadena de fluido no modela gas libre. **No se importan límites de operación** de otras fuentes: sin ellos no se puede decir que una condición sea «buena» o «mala».

In [14]:
from analisis.calidad import plausibilidad_operacion
pl = pd.DataFrame([f for h in hojas for f in plausibilidad_operacion(h)])
orden = [h["hoja"] for h in hojas]
pivote = pl.pivot(index=["control", "descripcion"], columns="hoja", values="resultado")[orden].reset_index()
no_cumple = int((pl["resultado"] == "NO CUMPLE").sum())
print("Controles ejecutados:", len(pl), "| incumplimientos:", no_cumple)
T(pivote, ancha=True)

Controles ejecutados: 60 | incumplimientos: 0


control,descripcion,QF-554H,QF-545H,QF-500H,QF-378HST,QF-549HST
D-01,Caudal de operación >= 0,cumple,cumple,cumple,cumple,cumple
D-02,Frecuencia de operación > 0,cumple,cumple,cumple,cumple,cumple
D-03,Presión de sensor > 0,cumple,cumple,cumple,cumple,cumple
D-04,THP >= 0,cumple,cumple,cumple,cumple,cumple
D-05,°API > -131.5 (denominador de F-01),cumple,cumple,cumple,cumple,cumple
D-06,BS&W entre 0 y 1 (la fórmula lo usa como fracción),cumple,cumple,cumple,cumple,cumple
D-07,ID del tubing > 0,cumple,cumple,cumple,cumple,cumple
D-08,Número de etapas > 0,cumple,cumple,cumple,cumple,cumple
D-09,Frecuencia de la curva > 0,cumple,cumple,cumple,cumple,cumple
G-01,TVD de descarga <= MD de descarga,cumple,cumple,cumple,cumple,cumple


**Interpretación.** No hay incumplimientos. La observación `O-01` indica que la presión del sensor supera la de burbuja en todas las hojas: eso es coherente con una cadena de fluido que **no modela gas libre**, pero es una observación, no una validación de la hipótesis monofásica. Un valor que **sí** hubiera fallado (por ejemplo `BS&W = 97.55`, que es como se leería «%») lo detecta `D-06`; esa detección está cubierta por una prueba automática (`tests/test_controles_y_mapeo.py`).

## Paso 9 · Criterios de aceptación y dictamen por hoja

In [15]:
crit = Criterios("01_inspeccion_y_calidad_datos")
sha_final = huellas(COPIA_TRABAJO)["sha256"]
crit.comprobar("C01-1", "5 hojas visibles y 1 270 fórmulas inventariadas",
               len(inv) == 5 and (inv["estado"] == "visible").all() and int(inv["formulas"].sum()) == 1270,
               f"hojas={len(inv)}, fórmulas={int(inv['formulas'].sum())}")
crit.comprobar("C01-2", "Rótulos clave hallados en todas las hojas", bool(anclas.drop(columns="hoja").notna().all().all()),
               "sin None en la tabla de anclas")
crit.comprobar("C01-3", "Ningún error guardado es precedente de THD ni de Q a producir",
               int(rastreo["errores entre precedentes"].sum()) == 0 and not rastreo["cita otra hoja"].any(),
               f"{len(errores)} errores; 0 en {len(rastreo)} cadenas rastreadas")
crit.comprobar("C01-4", "Cero datos requeridos faltantes", all(c["faltantes"] == "ninguna" for c in completitud),
               f"{len(requeridas)} variables x {len(hojas)} hojas")
crit.comprobar("C01-5", "Geometría del survey correcta en las 5 hojas",
               bool(geom[["md_estrictamente_creciente", "vertical_menor_o_igual_que_md", "inclinacion_en_0_180", "azimut_en_0_360"]].all().all())
               and int(geom["md_duplicadas"].sum()) == 0 and int(geom["valores_nulos"].sum()) == 0, "5/5 hojas")
crit.comprobar("C01-6", "Survey = curvatura mínima dentro de 0.01", bool(mcm["coincide_dentro_del_redondeo"].all()),
               f"máx ΔTVD={mcm['max_dif_tvd_ft'].max():.4f} ft; máx Δdogleg={mcm['max_dif_dogleg_deg_100ft'].max():.4f}")
crit.comprobar("C01-7", "TVD de equipos = interpolación del survey",
               bool(tvd_equipos["dentro_del_survey"].all()) and bool((tvd_equipos["dif_ft"].abs() < 1e-6).all()) and bool(tvd_equipos["tvd_menor_o_igual_que_md"].all()),
               f"{len(tvd_equipos)} comparaciones")
crit.comprobar("C01-8", "Ningún control de dominio incumplido", no_cumple == 0, f"{len(pl)} controles")
crit.comprobar("C01-9", "El libro no cambió durante el cuaderno", sha_inicial == sha_final, sha_final[:16] + "…")
T(crit.tabla())

id,criterio,resultado,evidencia
C01-1,5 hojas visibles y 1 270 fórmulas inventariadas,CUMPLE,"hojas=5, fórmulas=1270"
C01-2,Rótulos clave hallados en todas las hojas,CUMPLE,sin None en la tabla de anclas
C01-3,Ningún error guardado es precedente de THD ni de Q a producir,CUMPLE,13 errores; 0 en 10 cadenas rastreadas
C01-4,Cero datos requeridos faltantes,CUMPLE,19 variables x 5 hojas
C01-5,Geometría del survey correcta en las 5 hojas,CUMPLE,5/5 hojas
C01-6,Survey = curvatura mínima dentro de 0.01,CUMPLE,máx ΔTVD=0.0050 ft; máx Δdogleg=0.0050
C01-7,TVD de equipos = interpolación del survey,CUMPLE,15 comparaciones
C01-8,Ningún control de dominio incumplido,CUMPLE,60 controles
C01-9,El libro no cambió durante el cuaderno,CUMPLE,ccb5a738baea1332…


In [16]:
crit.exigir()
dictamen = []
for h in hojas:
    n = h["hoja"]
    g, m = geom[geom["hoja"] == n].iloc[0], mcm[mcm["hoja"] == n].iloc[0]
    t = tvd_equipos[tvd_equipos["hoja"] == n]
    en_cadena = int(rastreo.loc[rastreo["hoja"] == n, "errores entre precedentes"].sum())
    sin_incumplimientos = not (pl[(pl["hoja"] == n)]["resultado"] == "NO CUMPLE").any()
    apta = (en_cadena == 0 and bool(m["coincide_dentro_del_redondeo"]) and bool(t["dif_ft"].abs().lt(1e-6).all())
            and bool(g["md_estrictamente_creciente"]) and sin_incumplimientos)
    dictamen.append({
        "hoja": n, "pozo": h["pozo"],
        "errores guardados": int((errores["hoja"] == n).sum()),
        "errores en la cadena": en_cadena,
        "survey = curvatura mínima": bool(m["coincide_dentro_del_redondeo"]),
        "TVD de equipos = survey": bool(t["dif_ft"].abs().lt(1e-6).all()),
        "dictamen": "Apta para reproducir el cálculo de la hoja; no para decidir" if apta else "NO APTA: ver controles",
    })
T(pd.DataFrame(dictamen))

hoja,pozo,errores guardados,errores en la cadena,survey = curvatura mínima,TVD de equipos = survey,dictamen
QF-554H,QF-084,0,0,sí,sí,Apta para reproducir el cálculo de la hoja; no para decidir
QF-545H,QF-301,0,0,sí,sí,Apta para reproducir el cálculo de la hoja; no para decidir
QF-500H,QF-147,0,0,sí,sí,Apta para reproducir el cálculo de la hoja; no para decidir
QF-378HST,QF-005,0,0,sí,sí,Apta para reproducir el cálculo de la hoja; no para decidir
QF-549HST,QF-306,13,0,sí,sí,Apta para reproducir el cálculo de la hoja; no para decidir


## Conclusión técnica

- **Resultado:** el libro es estructuralmente sano para el propósito del expediente. Las cinco hojas son cálculos autónomos; los 13 `#DIV/0!` están fuera de la cadena; el survey es coherente con curvatura mínima; las TVD de los equipos provienen del survey.
- **Defectos reales que sí afectan el uso:** (1) las fechas `=TODAY()` impiden análisis temporal; (2) las posiciones cambian entre hojas, así que **toda lectura debe ser por rótulo**; (3) los 13 errores deben comunicarse a quien mantiene el libro.
- **Lo que este cuaderno no puede decir:** que los valores medidos (presiones, caudales) sean correctos; solo que están completos, en dominio y que las profundidades son internamente consistentes.
- **Dictamen:** «apta para reproducir el cálculo de la hoja». La frase **no** equivale a «apta para decidir»: las reservas físicas (BS&W, signo de PIP, constantes) se tratan en los cuadernos 02 a 05.
- **Siguiente paso:** cuaderno 02, trazabilidad de las fórmulas.

## Relación con el sistema de reglas

Dictamina qué defectos contaminan el cálculo y cuáles no. No crea umbrales. El motor usa esa distinción cuando un dato falta o cuando BS&W sale de 0 a 1: advierte, no inventa el valor.

## Limitaciones

- La calidad se mide con controles **propios** (tolerancias, plausibilidad física, curvatura mínima): son criterios del expediente, no normas.
- La curvatura mínima es un control **independiente** de las columnas del survey, no un reemplazo: una desviación indica inconsistencia, no cuál de los dos es el correcto.
- Una hoja sin hallazgos significa «sin hallazgos con estos controles».
- El rastreo de precedentes resuelve referencias A1 de la misma hoja; una referencia dinámica (INDIRECT, OFFSET) no se seguiría (no hay en la cadena).

## Referencias

Solo se listan las fuentes realmente consultadas en este cuaderno.

1. **Libro de trabajo:** `entrada/BES_jun.xlsx`, copia byte a byte de «Copia de FINAL BES-jun.xlsx» (SHA-256 `ccb5a738baea13321993e8b6eccefd6cac90ecda86b1c4b9181f897fcfa83f85`). Cinco hojas: QF-554H, QF-545H, QF-500H, QF-378HST y QF-549HST.
2. **Software:** Python 3.14.8, pandas 3.0.6, numpy 2.5.4, matplotlib 3.11.2, openpyxl 3.1.5 (versiones tomadas de la ficha de entorno de cada ejecución).
3. **Métodos de uso general sin cita verificada en las fuentes del expediente:** curvatura mínima (control de survey) y relación hidrostática P = ρ·g·h. Se aplican como están escritos en la hoja o se describen como método estándar, sin atribuirlos a un texto concreto.